# 00 — Data Preparation
### Shared preprocessing pipeline for all models
Run this notebook FIRST before any model notebook.

**Prepares data for:**
- Forecasting models (LSTM, GRU, TFT) — predicts future breakdown
- Classification model (FAG-TFT) — classifies current state
- Anomaly Gate — detects unknown faults


In [4]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('✅ Imports complete.')

✅ Imports complete.


In [5]:
# FILE PATH
DATA_FILE = r'C:\Users\DeelakaD\OneDrive - MAS Holdings (Pvt) Ltd\Documents\GitHub\Preventing-Mechanism\Datasets\Combined_Machine_Data_Shuffled.xlsx'

TIME_STEPS = 5
H = 5  # Forecasting horizon: predict if breakdown in next 5 events

KNOWN_BREAKDOWNS = [
    'Needle Breakages',
    'High Foot Pressure',
    'Cut/Needle Hole',
    'Thread Breakages',
    'Pneumatic Issues',
    'Thread Jamming',
    'Code Uneven',
    'Roping',
    'Oil Mark',
    'Skip Stitches/Slip',
    'Gathering/Puckering',
    'Waveness',
    'Binding/Seam Open',
    'Blade Blunt',
]
ALLOWED_STATES = ['Healthy'] + KNOWN_BREAKDOWNS

# FREQUENCY BAND GROUPS (for FAG-TFT)
VIB_GROUP_1 = [f'{i}-{i+10}Hz' for i in range(10,  100, 10)]   # 10-100Hz
VIB_GROUP_2 = [f'{i}-{i+10}Hz' for i in range(100, 300, 10)]   # 100-300Hz
VIB_GROUP_3 = [f'{i}-{i+10}Hz' for i in range(300, 610, 10)]   # 300-610Hz
ELEC_FEATS  = ['machineVoltageMean','machineVoltageMin','machineVoltageMax',
               'machineCurrentMean','machineCurrentMin','machineCurrentMax']

print(f'✅ Config loaded.')
print(f'   TIME_STEPS             : {TIME_STEPS}')
print(f'   Forecasting horizon H  : {H}')
print(f'   Known breakdowns       : {len(KNOWN_BREAKDOWNS)}')
print(f'   VIB Group 1 (10-100Hz) : {len(VIB_GROUP_1)} bands')
print(f'   VIB Group 2 (100-300Hz): {len(VIB_GROUP_2)} bands')
print(f'   VIB Group 3 (300-610Hz): {len(VIB_GROUP_3)} bands')
print(f'   Electrical features    : {len(ELEC_FEATS)}')


✅ Config loaded.
   TIME_STEPS             : 5
   Forecasting horizon H  : 5
   Known breakdowns       : 14
   VIB Group 1 (10-100Hz) : 9 bands
   VIB Group 2 (100-300Hz): 20 bands
   VIB Group 3 (300-610Hz): 31 bands
   Electrical features    : 6


In [6]:
# LOAD DATA
master_df = pd.read_excel(DATA_FILE)

# Filter valid vibration records (must start with '10')
master_df = master_df[master_df['machineVibration'].astype(str).str.startswith('10')].copy()

# Fill empty Breakdown column as 'Healthy'
master_df['Breakdown'] = master_df['Breakdown'].fillna('Healthy')

# Keep only allowed states
master_df = master_df[master_df['Breakdown'].isin(ALLOWED_STATES)].reset_index(drop=True)

# Sort by dateTime (chronological order)
master_df['dateTime'] = pd.to_datetime(master_df['dateTime'])
master_df = master_df.sort_values('dateTime').reset_index(drop=True)

# Create time_gap: seconds between consecutive pedal presses (capped at 300s)
master_df['time_gap'] = master_df['dateTime'].diff().dt.total_seconds().fillna(0).clip(upper=300)

print(f'✅ Data loaded: {len(master_df)} total records')
print(f'   Date range: {master_df["dateTime"].iloc[0]} to {master_df["dateTime"].iloc[-1]}')
print(f'\nClass distribution:')
print(master_df['Breakdown'].value_counts())


✅ Data loaded: 5378 total records
   Date range: 2025-06-01 06:00:00 to 2025-06-20 07:10:21.905000

Class distribution:
Breakdown
Healthy               4947
High Foot Pressure     110
Blade Blunt            107
Skip Stitches/Slip     107
Waveness               107
Name: count, dtype: int64


In [7]:
# FEATURE EXTRACTION (67 features: 60 vib + 6 elec + 1 time_gap)
def extract_features(df):
    vib_records = []
    for val in df['machineVibration']:
        vib_dict = {}
        parts = str(val).split(',')
        try:
            for i in range(0, len(parts)-1, 2):
                f_start = int(parts[i])
                vib_dict[f'{f_start}-{f_start+10}Hz'] = int(parts[i+1])
        except: pass
        vib_records.append(vib_dict)

    expected_vib = [f'{i}-{i+10}Hz' for i in range(10, 610, 10)]
    vib_df  = pd.DataFrame(vib_records).reindex(columns=expected_vib, fill_value=0)
    elec_df = df[ELEC_FEATS].ffill().fillna(0).reset_index(drop=True)
    time_gap_df = df[['time_gap']].reset_index(drop=True)
    return pd.concat([vib_df.reset_index(drop=True), elec_df, time_gap_df], axis=1)

X_all = extract_features(master_df)
y_all = master_df['Breakdown'].values
print(f'✅ Features extracted: {X_all.shape}  (60 vib + 6 elec + 1 time_gap = 67 features)')


✅ Features extracted: (5378, 67)  (60 vib + 6 elec + 1 time_gap = 67 features)


---
## Classification Data (for FAG-TFT + Anomaly Gate)


In [8]:
# ENCODE LABELS FOR CLASSIFICATION
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_all)
num_classes = len(encoder.classes_)
print(f'✅ Classes: {list(encoder.classes_)}')

# TRAIN/TEST SPLIT
# Converts pandas DataFrame to numpy array
# Stratify ensures both sets maintain the same class proportions as the original data.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_all.values, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

# SCALE (fit on train only)
scaler = StandardScaler()
# fit - the scaler calculates mean and std for each feature
# transform - applies the formula to every value
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)

print(f'   Train: {X_train_scaled.shape}  Test: {X_test_scaled.shape}')


✅ Classes: ['Blade Blunt', 'Healthy', 'High Foot Pressure', 'Skip Stitches/Slip', 'Waveness']
   Train: (4302, 67)  Test: (1076, 67)


In [9]:
# SEQUENCE CREATION FOR CLASSIFICATION
def create_sequences(X, y, time_steps):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i+time_steps])
        ys.append(y[i+time_steps])
    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train, TIME_STEPS)
X_test_seq,  y_test_seq  = create_sequences(X_test_scaled,  y_test,  TIME_STEPS)

print(f'✅ Classification sequences created.')
print(f'   Train: {X_train_seq.shape}  →  (samples, {TIME_STEPS}, 67)')
print(f'   Test : {X_test_seq.shape}')


✅ Classification sequences created.
   Train: (4297, 5, 67)  →  (samples, 5, 67)
   Test : (1071, 5, 67)


In [10]:
# HEALTHY-ONLY DATA FOR ANOMALY GATE
healthy_mask = master_df['Breakdown'].values == 'Healthy'
X_healthy    = extract_features(master_df[healthy_mask])
X_healthy_scaled = scaler.transform(X_healthy.values)

def create_sequences_X(X, time_steps):
    return np.array([X[i:i+time_steps] for i in range(len(X)-time_steps)])

X_healthy_seq = create_sequences_X(X_healthy_scaled, TIME_STEPS)
split = int(len(X_healthy_seq)*0.8)
X_ae_train, X_ae_test = X_healthy_seq[:split], X_healthy_seq[split:]

print(f'✅ Autoencoder sequences: train={X_ae_train.shape}  test={X_ae_test.shape}')


✅ Autoencoder sequences: train=(3953, 5, 67)  test=(989, 5, 67)


---
## Forecasting Data (for LSTM, GRU, TFT Forecaster)


In [11]:
# CREATE FORECASTING LABELS
# breakdown_now: is this record a breakdown?
master_df['breakdown_now'] = (master_df['Breakdown'] != 'Healthy').astype(int)

# future_breakdown: will a breakdown happen in the next H events?
# future_reason: which type of breakdown is coming?
future_breakdown = []
future_reason = []

for i in range(len(master_df)):
    future_window = master_df.iloc[i+1:i+1+H]
    bd_in_future = future_window[future_window['Breakdown'] != 'Healthy']
    if len(bd_in_future) > 0:
        future_breakdown.append(1)
        future_reason.append(bd_in_future.iloc[0]['Breakdown'])
    else:
        future_breakdown.append(0)
        future_reason.append('No Breakdown')

master_df['future_breakdown'] = future_breakdown
master_df['future_reason'] = future_reason

# Remove rows where breakdown is already happening (can't forecast what already happened)
df_forecast = master_df[master_df['breakdown_now'] == 0].copy().reset_index(drop=True)

print(f'✅ Forecasting labels created.')
print(f'   Total records (active breakdowns removed): {len(df_forecast)}')
print(f'   Safe (no breakdown coming)   : {(df_forecast["future_breakdown"]==0).sum()}')
print(f'   Approaching breakdown        : {(df_forecast["future_breakdown"]==1).sum()}')
print(f'\n   Approaching breakdown by type:')
print(df_forecast[df_forecast['future_breakdown']==1]['future_reason'].value_counts().to_string())


✅ Forecasting labels created.
   Total records (active breakdowns removed): 4947
   Safe (no breakdown coming)   : 4631
   Approaching breakdown        : 316

   Approaching breakdown by type:
future_reason
Blade Blunt           80
Skip Stitches/Slip    80
High Foot Pressure    79
Waveness              77


In [12]:
# PREPARE FORECASTING SEQUENCES
X_forecast = extract_features(df_forecast)
y_forecast_binary = df_forecast['future_breakdown'].values

# Encode breakdown reason for TFT type prediction
encoder_reason = LabelEncoder()
y_forecast_reason = encoder_reason.fit_transform(df_forecast['future_reason'].values)
num_reason_classes = len(encoder_reason.classes_)

# Train/test split for forecasting
X_fc_train_raw, X_fc_test_raw, y_fc_train_bin, y_fc_test_bin, y_fc_train_type, y_fc_test_type = train_test_split(
    X_forecast.values, y_forecast_binary, y_forecast_reason,
    test_size=0.2, random_state=42, stratify=y_forecast_binary)

# Scale using SAME scaler as classification (consistency across all models)
X_fc_train_scaled = scaler.transform(X_fc_train_raw)
X_fc_test_scaled  = scaler.transform(X_fc_test_raw)

# Create sequences
X_fc_train_seq, y_fc_train_bin_seq = create_sequences(X_fc_train_scaled, y_fc_train_bin, TIME_STEPS)
X_fc_test_seq,  y_fc_test_bin_seq  = create_sequences(X_fc_test_scaled,  y_fc_test_bin,  TIME_STEPS)

# Type sequences (same X, different y)
_, y_fc_train_type_seq = create_sequences(X_fc_train_scaled, y_fc_train_type, TIME_STEPS)
_, y_fc_test_type_seq  = create_sequences(X_fc_test_scaled,  y_fc_test_type,  TIME_STEPS)

print(f'✅ Forecasting sequences created.')
print(f'   Train: {X_fc_train_seq.shape}')
print(f'   Test : {X_fc_test_seq.shape}')
print(f'   Binary class counts (train): {np.bincount(y_fc_train_bin_seq)}')
print(f'   Reason classes: {list(encoder_reason.classes_)}')


✅ Forecasting sequences created.
   Train: (3952, 5, 67)
   Test : (985, 5, 67)
   Binary class counts (train): [3700  252]
   Reason classes: ['Blade Blunt', 'High Foot Pressure', 'No Breakdown', 'Skip Stitches/Slip', 'Waveness']


---
## Save All Prepared Data


In [13]:
# SAVE CLASSIFICATION DATA
prepared_classification = {
    'X_train_seq'  : X_train_seq,
    'X_test_seq'   : X_test_seq,
    'y_train_seq'  : y_train_seq,
    'y_test_seq'   : y_test_seq,
    'X_ae_train'   : X_ae_train,
    'X_ae_test'    : X_ae_test,
    'num_classes'  : num_classes,
    'num_features' : X_train_seq.shape[2],
    'TIME_STEPS'   : TIME_STEPS,
    'VIB_GROUP_1'  : VIB_GROUP_1,
    'VIB_GROUP_2'  : VIB_GROUP_2,
    'VIB_GROUP_3'  : VIB_GROUP_3,
    'ELEC_FEATS'   : ELEC_FEATS,
    'feature_names': list(X_all.columns),
}

# SAVE FORECASTING DATA
prepared_forecasting = {
    'X_fc_train_seq'      : X_fc_train_seq,
    'X_fc_test_seq'       : X_fc_test_seq,
    'y_fc_train_bin_seq'  : y_fc_train_bin_seq,
    'y_fc_test_bin_seq'   : y_fc_test_bin_seq,
    'y_fc_train_type_seq' : y_fc_train_type_seq,
    'y_fc_test_type_seq'  : y_fc_test_type_seq,
    'num_features'        : X_fc_train_seq.shape[2],
    'num_reason_classes'  : num_reason_classes,
    'TIME_STEPS'          : TIME_STEPS,
    'H'                   : H,
    'feature_names'       : list(X_forecast.columns),
}

with open('prepared_classification.pkl','wb') as f: pickle.dump(prepared_classification, f)
with open('prepared_forecasting.pkl','wb') as f: pickle.dump(prepared_forecasting, f)  # for LSTM, GRU, CNN LSTM(training and testing sequences)
with open('scaler.pkl','wb') as f: pickle.dump(scaler, f) # this carrries training mean and std for Live Analyser file
with open('encoder.pkl','wb') as f: pickle.dump(encoder, f) # this carries the mapping of class labels to integers for Live Analyser file
with open('encoder_reason.pkl','wb') as f: pickle.dump(encoder_reason, f)

print('✅ Saved:')
print('   prepared_classification.pkl  — for FAG-TFT + Anomaly Gate')
print('   prepared_forecasting.pkl     — for LSTM, GRU, TFT Forecaster')
print('   scaler.pkl                   — shared scaler')
print('   encoder.pkl                  — classification label encoder')
print('   encoder_reason.pkl           — forecasting reason encoder')
print(f'\n   Classification: {num_classes} classes, {X_train_seq.shape[2]} features, TIME_STEPS={TIME_STEPS}')
print(f'   Forecasting: binary + {num_reason_classes} reason classes, {X_fc_train_seq.shape[2]} features, H={H}')


✅ Saved:
   prepared_classification.pkl  — for FAG-TFT + Anomaly Gate
   prepared_forecasting.pkl     — for LSTM, GRU, TFT Forecaster
   scaler.pkl                   — shared scaler
   encoder.pkl                  — classification label encoder
   encoder_reason.pkl           — forecasting reason encoder

   Classification: 5 classes, 67 features, TIME_STEPS=5
   Forecasting: binary + 5 reason classes, 67 features, H=5
